# Master Project: AI Insurance Adjuster
## YOLOv8 Training on D-Fire Dataset (Fire & Smoke Detection)

Run this notebook in Google Colab with GPU runtime: `Runtime` -> `Change runtime type` -> `T4 GPU`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload master_project.zip to Google Drive, then uncomment:
# !unzip -q /content/drive/MyDrive/master_project.zip -d /content/master_project

# Clone from GitHub:
!git clone https://github.com/Oussama1317/master_project.git /content/master_project

import os
os.makedirs('/content/master_project', exist_ok=True)
%cd /content/master_project

In [ ]:
!pip install -q ultralytics kagglehub

In [ ]:
import kagglehub
import shutil
from pathlib import Path

DATA_DIR = Path("data/raw/dfire")
if not DATA_DIR.exists():
    print("Downloading D-Fire dataset...")
    path = Path(kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo"))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for item in path.iterdir():
        dest = DATA_DIR / item.name
        if dest.exists():
            if item.is_dir():
                shutil.rmtree(str(dest))
            else:
                dest.unlink()
        if item.is_dir():
            shutil.copytree(str(item), str(dest))
        else:
            shutil.copy2(str(item), str(dest))
    print("Download complete!")
else:
    print("Dataset already exists")

# Rewrite data.yaml — use absolute path to avoid any resolution issues
yaml_path = DATA_DIR / "data.yaml"
abs_root = str(DATA_DIR.resolve())
yaml_path.write_text(f"path: {abs_root}\ntrain: data/train/images\nval: data/val/images\ntest: data/test/images\nnc: 2\nnames: ['smoke', 'fire']\n")
print(f"data.yaml rewritten with absolute path: {abs_root}")

train_imgs = list((DATA_DIR/'data'/'train'/'images').iterdir()) if (DATA_DIR/'data'/'train'/'images').exists() else []
print(f"Dataset ready: {len(train_imgs)} train images")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="data/raw/dfire/data.yaml",
    epochs=50,
    batch=32,
    imgsz=640,
    patience=15,
    device="cuda",
    workers=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    project="runs/train",
    name="dfire_full",
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

In [ ]:
# Validate best model on test set
from ultralytics import YOLO
best_path = "runs/train/dfire_full/weights/best.pt"
val_model = YOLO(best_path)
metrics = val_model.val(data="data/raw/dfire/data.yaml", split="test", device="cuda")
print(f"\nResults:\nmAP50-95: {metrics.box.map:.4f}\nmAP50: {metrics.box.map50:.4f}\nPrecision: {metrics.box.p:.4f}\nRecall: {metrics.box.r:.4f}")

In [ ]:
# Save results to Google Drive
import os
os.makedirs('/content/drive/MyDrive/master_project_results', exist_ok=True)
!cp -r runs/train/dfire_full /content/drive/MyDrive/master_project_results/

In [ ]:
# Download best.pt locally via browser
from google.colab import files
files.download("runs/train/dfire_full/weights/best.pt")